In [2]:
import pandas as pd
import os

# ==========================================
# 1. 基础配置：请修改为你电脑上的实际路径
# ==========================================
file1_path = r'D:\数据\业绩\处理后\sheet1.csv'
file2_path = r'D:\数据\业绩\处理后\sheet2.csv'
output_path = r'D:\数据\业绩\处理后\FundPerform.csv'

# ==========================================
# 2. 读取与合并数据 (分块读取以防内存溢出)
# ==========================================
print("正在读取并合并数据...")
# 先读取第一个文件 (Access 导出的 CSV 通常是 gbk 编码)
df1 = pd.read_csv(file1_path, encoding='gbk')

# 分块读取第二个文件并追加，这样对 1GB+ 的数据更安全
reader = pd.read_csv(file2_path, encoding='gbk', chunksize=100000)
for chunk in reader:
    df1 = pd.concat([df1, chunk], ignore_index=True)

# 将合并后的变量命名为 FundPerform
FundPerform = df1
print(f"合并完成，当前总行数: {len(FundPerform)}")

# ==========================================
# 3. 关键格式修复：修复基金代码 000001 问题
# ==========================================
print("正在修复基金代码格式...")
# 强制转换为 6 位字符串，不足的前面补 0
FundPerform['Symbol'] = (
    FundPerform['Symbol']
    .astype(str)
    .str.split('.').str[0]  # 防止出现 "1.0"
    .str.zfill(6)           # 补齐为 "000001"
)

# ==========================================
# 4. 生成“排名/总数”拼接列，并动态调整位置
# ==========================================
# 定义需要处理的指标前缀
metrics = [ 'TMTiming', 'TMStkSlct']

# 自定义格式化函数：彻底去掉 .0 并处理空值
def format_rank_str(x):
    if pd.isna(x) or str(x).lower() == 'nan' or str(x) == '':
        return ""
    try:
        return str(int(float(x)))
    except:
        return ""

for m in metrics:
    rnk_col = f"{m}Rnk"
    total_col = f"NumberToRnk{m}"
    new_col = f"{m}_FullRnk"

    if rnk_col in FundPerform.columns and total_col in FundPerform.columns:
        print(f"处理指标布局: {m}...")

        # 拼接字符串
        FundPerform[new_col] = (
            FundPerform[rnk_col].map(format_rank_str) +
            "/" +
            FundPerform[total_col].map(format_rank_str)
        )

        # 清理只有斜杠的无效数据
        FundPerform.loc[FundPerform[new_col] == "/", new_col] = ""

        # 删除原始的排名和总数列（节省内存）
        FundPerform.drop(columns=[rnk_col, total_col], inplace=True)

# ==========================================
# 5. 强制列顺序重排：让排名紧跟在数值后面
# ==========================================
print("正在优化列顺序布局...")
all_cols = FundPerform.columns.tolist()
new_order = []
seen_cols = set()

# A. 放入基础信息列
base_info = ['TradingDate', 'Symbol', 'FullName']
for col in base_info:
    if col in all_cols:
        new_order.append(col)
        seen_cols.add(col)

# B. 按顺序排放 [数值, 排名/总数]
for m in metrics:
    if m in all_cols:
        new_order.append(m)
        seen_cols.add(m)

    f_rnk = f"{m}_FullRnk"
    if f_rnk in all_cols:
        new_order.append(f_rnk)
        seen_cols.add(f_rnk)

# C. 将剩下的其他列追加到最后
for col in all_cols:
    if col not in seen_cols:
        new_order.append(col)

# 应用新顺序
FundPerform = FundPerform[new_order]

# ==========================================
# 6. 保存最终结果并验证
# ==========================================
print(f"正在保存最终文件至: {output_path}")
# 使用 utf-8-sig 编码，确保 Excel 打开中文不乱码
FundPerform.to_csv(output_path, index=False, encoding='utf-8-sig')

print("\n--- 全部处理成功！ ---")

正在读取并合并数据...
合并完成，当前总行数: 1629708
正在修复基金代码格式...
处理指标布局: TMTiming...
处理指标布局: TMStkSlct...
正在优化列顺序布局...
正在保存最终文件至: D:\数据\业绩\处理后\FundPerform.csv

--- 全部处理成功！ ---
